In [2]:
import random
import pandas as pd
from datetime import date, timedelta

random.seed(42)

# name, promised days, actual avg days, spread (variability), unit cost
vendors = [
    ("V1", 5, 5, 1, 100),    # reliable
    ("V2", 7, 8, 4, 90),     # thoda late, unpredictable
    ("V3", 10, 10, 2, 80),   # slow but consistent
    ("V4", 6, 7, 3, 95),
    ("V5", 12, 15, 6, 70),   # cheap but risky
    ("V6", 8, 8, 1, 85),     # reliable
]

start_date = date(2024, 1, 1)

rows = []
for i in range(5000):
    v = random.choice(vendors)

    order_date = start_date + timedelta(days=random.randint(0, 729))
    promised_lt = v[1]
    actual_lt = round(random.gauss(v[2], v[3]))

    if actual_lt < 1:
        actual_lt = 1

    promised_date = order_date + timedelta(days=promised_lt)
    delivery_date = order_date + timedelta(days=actual_lt)
    qty = random.randint(100, 1000)

    rows.append([i + 1, v[0], order_date, promised_date,
                 delivery_date, qty, v[4]])

df = pd.DataFrame(rows, columns=["order_id", "vendor", "order_date",
                                 "promised_date", "delivery_date",
                                 "qty", "unit_cost"])
df.to_csv("orders.csv", index=False)

# Quick check: har vendor ka summary
for name in ["V1", "V2", "V3", "V4", "V5", "V6"]:
    sub = df[df["vendor"] == name]
    lt_list = []
    late_count = 0
    for idx, row in sub.iterrows():
        lt = (row["delivery_date"] - row["order_date"]).days
        lt_list.append(lt)
        if row["delivery_date"] > row["promised_date"]:
            late_count = late_count + 1
    avg_lt = sum(lt_list) / len(lt_list)
    on_time_pct = (len(sub) - late_count) / len(sub) * 100
    print(name, "orders:", len(sub), "avg LT:", round(avg_lt, 1),
          "on-time %:", round(on_time_pct, 1))

V1 orders: 845 avg LT: 5.1 on-time %: 65.1
V2 orders: 810 avg LT: 8.1 on-time %: 43.6
V3 orders: 811 avg LT: 10.1 on-time %: 59.2
V4 orders: 827 avg LT: 7.0 on-time %: 44.3
V5 orders: 827 avg LT: 15.4 on-time %: 31.0
V6 orders: 880 avg LT: 7.9 on-time %: 70.1


In [5]:
print(len(df))
print(df)

5000
      order_id vendor  order_date promised_date delivery_date  qty  unit_cost
0            1     V6  2024-04-24    2024-05-02    2024-05-03  328         85
1            2     V2  2024-04-14    2024-04-21    2024-04-23  792         90
2            3     V6  2025-07-12    2025-07-20    2025-07-21  130         85
3            4     V1  2024-08-11    2024-08-16    2024-08-17  338        100
4            5     V5  2025-09-08    2025-09-20    2025-09-27  765         70
...        ...    ...         ...           ...           ...  ...        ...
4995      4996     V4  2024-12-19    2024-12-25    2024-12-27  751         95
4996      4997     V3  2024-03-31    2024-04-10    2024-04-08  642         80
4997      4998     V2  2025-02-18    2025-02-25    2025-03-02  486         90
4998      4999     V5  2024-05-05    2024-05-17    2024-05-20  375         70
4999      5000     V5  2024-08-21    2024-09-02    2024-09-08  429         70

[5000 rows x 7 columns]


In [6]:
pip install mysql-connector-python

   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
    --------------------------------------- 0.3/17.7 MB ? eta -:--:--
    --------------------------------------- 0.3/17.7 MB ? eta -:--:--
    --------------------------------------- 0.3/17.7 MB ? eta -:--:--
    --------------------------------------- 0.3/17.7 MB ? eta -:--:--
    --------------------------------------- 0.3/17.7 MB ? eta -:--:--
    ----------------

In [9]:
import pandas as pd
import mysql.connector

df = pd.read_csv("orders.csv")

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="YOUR_PASSWORD",
    database="vendor_project"
)
cursor = conn.cursor()

# pehle table khali karo, taaki duplicate na ho
cursor.execute("DELETE FROM orders")

query = """INSERT INTO orders
(order_id, vendor, order_date, promised_date, delivery_date, qty, unit_cost)
VALUES (%s, %s, %s, %s, %s, %s, %s)"""

for i in range(len(df)):
    row = df.iloc[i]
    values = (int(row["order_id"]), row["vendor"], row["order_date"],
              row["promised_date"], row["delivery_date"],
              int(row["qty"]), int(row["unit_cost"]))
    cursor.execute(query, values)

conn.commit()

cursor.execute("SELECT COUNT(*) FROM orders")
print("Rows in table:", cursor.fetchone()[0])

cursor.close()
conn.close()

Rows in table: 5000


In [11]:
import pandas as pd
import math

df = pd.read_csv("orders.csv", parse_dates=["order_date", "promised_date", "delivery_date"])

Z = 1.65  # 95% service level
total_days = 730  # data 2 saal ka hai

results = []

for name in ["V1", "V2", "V3", "V4", "V5", "V6"]:
    sub = df[df["vendor"] == name]

    lt_list = []
    late_count = 0
    total_qty = 0

    for idx, row in sub.iterrows():
        lt = (row["delivery_date"] - row["order_date"]).days
        lt_list.append(lt)
        total_qty = total_qty + row["qty"]
        if row["delivery_date"] > row["promised_date"]:
            late_count = late_count + 1

    n = len(lt_list)

    # mean
    total = 0
    for x in lt_list:
        total = total + x
    mean_lt = total / n

    # standard deviation
    sq_sum = 0
    for x in lt_list:
        sq_sum = sq_sum + (x - mean_lt) ** 2
    std_lt = math.sqrt(sq_sum / (n - 1))

    on_time_pct = (n - late_count) / n * 100
    daily_demand = total_qty / total_days

    safety_stock = Z * std_lt * daily_demand
    reorder_point = daily_demand * mean_lt + safety_stock

    results.append([name, n, round(mean_lt, 1), round(std_lt, 2),
                    round(on_time_pct, 1), round(daily_demand),
                    round(safety_stock), round(reorder_point)])

summary = pd.DataFrame(results, columns=["vendor", "orders", "avg_lt", "std_lt",
                                         "on_time_pct", "daily_demand",
                                         "safety_stock", "reorder_point"])
print(summary)
summary.to_csv("vendor_summary.csv", index=False)

  vendor  orders  avg_lt  std_lt  on_time_pct  daily_demand  safety_stock  \
0     V1     845     5.1    1.04         65.1           639          1103   
1     V2     810     8.1    3.73         43.6           608          3739   
2     V3     811    10.1    1.96         59.2           601          1943   
3     V4     827     7.0    2.90         44.3           617          2955   
4     V5     827    15.4    6.06         31.0           641          6406   
5     V6     880     7.9    1.03         70.1           675          1147   

   reorder_point  
0           4363  
1           8692  
2           7986  
3           7296  
4          16285  
5           6509  


In [12]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\SUHANI SHARMA
['.anaconda', '.anyLogistix3ple', '.cache', '.conda', '.continuum', '.eclipse', '.gemini', '.ibm_cplex_studio', '.ipynb_checkpoints', '.ipython', '.jupyter', '.matplotlib', '.ms-ad', '.spyder-py3', '.streamlit', '.vscode', '100-days-ml', '100-days-of-ml', 'anaconda_projects', 'AppData', 'Application Data', 'Contacts', 'Cookies', 'CrossDevice', 'dataproject', 'Desktop.lnk', 'dlmgr_.pro', 'Documents', 'DownloadDirector', 'Downloads', 'Favorites', 'InstallAnywhere', 'Jedi', 'Links', 'Local Settings', 'Music', 'My Documents', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{4eb5ca1d-99ae-11ef-adb3-28d04367e210}.TM.blf', 'NTUSER.DAT{4eb5ca1d-99ae-11ef-adb3-28d04367e210}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{4eb5ca1d-99ae-11ef-adb3-28d04367e210}.TMContainer00000000000000000002.regtrans-ms', 'ntuser.ini', 'OneDrive', 'opl', 'orders.csv', 'PrintHood', 'PyCharmMiscProject', 'PycharmProjects', 'Recent', 'Saved Games', 'Searches